In [25]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 10000
#eval_interval = 2500
learning_rate = 3e-4

cpu


In [26]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™']


In [27]:
string_to_int = {ch:i for i, ch in enumerate(chars) }
int_to_string = {i:ch for i, ch in enumerate(chars) }

#Initialize Encoder and Decoder
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72, 60,
        63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73, 78,
        66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,  1,
        67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63, 31,
        73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77, 63,
         1, 73, 64,  1, 59, 72, 83, 73, 72, 63])


In [28]:
#Get training values
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1]for i in ix])
    #Push to the GPU
    x,y = x.to(device), y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs:')

#Print(x.shape)
print(x)
print('targets:')
print(y)


inputs:
tensor([[73, 76,  1, 77, 73, 71, 63, 78],
        [66, 67, 71, 15,  3,  0,  0, 49],
        [63,  1, 63, 72, 78, 67, 76, 63],
        [81, 59, 77,  1, 68, 63, 76, 69]])
targets:
tensor([[76,  1, 77, 73, 71, 63, 78, 66],
        [67, 71, 15,  3,  0,  0, 49, 66],
        [ 1, 63, 72, 78, 67, 76, 63,  1],
        [59, 77,  1, 68, 63, 76, 69, 63]])


In [29]:

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is", context, 'target is', target)

when input is tensor([49]) target is tensor(66)
when input is tensor([49, 66]) target is tensor(63)
when input is tensor([49, 66, 63]) target is tensor(1)
when input is tensor([49, 66, 63,  1]) target is tensor(45)
when input is tensor([49, 66, 63,  1, 45]) target is tensor(76)
when input is tensor([49, 66, 63,  1, 45, 76]) target is tensor(73)
when input is tensor([49, 66, 63,  1, 45, 76, 73]) target is tensor(68)
when input is tensor([49, 66, 63,  1, 45, 76, 73, 68]) target is tensor(63)


In [30]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embeddings_table = nn.Embedding(vocab_size, vocab_size
        #This is basically a lookup table of the token and the scores to
        #what follows it

        #An attention layer would be utilized with:
        #nn.Embedding(vocab_size, n_embd) plus a lm_head

    def forward(self, index, targets=None):
        #This is a row lookup
        #Index is (B,T) of integers.
        #Each integer gets replaced by its row. Output is (B,T,C) where C = vocab_size.
        #For each B sequences, for each of T positions, get a full score vector over the vocab
        logits = self.token_embeddings_table(index)
        

        if targets is None:
            loss = None
        else:
            #F.cross_entropy wants predictions as (N,C) and targets as (N,)
            #A flat list of N independent predictions
            #Currently we have a 3D block, so we need to collapse batch and time into one axis
            #In this case it is: B*T separate predictions each over C classes.
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get the predictions
            logits, loss = self.forward(index)

            #Focus only on the last time step
            logits = logits[:,-1,:] #becomes (B,C)

            #apply softmax to get the probability distribution
            #dim=-1 normalizes across the C axis so each row sums to 1
            #And becomes a valid probability distribution
            #If it was dim=0 it'd normalize across the batch.
            probs = F.softmax(logits, dim=-1) # (B,C)

            #sample from the distribution
            #torch.multinomial samples rather than taking argmax.
            #Argmax would be deterministic and collapse into loops
            #Sampling makes the output varied
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            #append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
    


F:ISw6rdW-O(Ybs5?“Gk™’B”kcQMW•[v!Ol9’7:(DV]fKgpX3&No!q’7(IvSekyVM8S-YMA1•8A1[wk3,adl[7]vCn.HB)V(;”s_Y&]BDneo!%IV3,:yi[7:&]N2Hp6nzu/h0m6bykZ/Mh”0—F::KZEh+sB;:A?zzbOHFC?6ot+#™iYJgL7A"L7.Ro] ]b6F+ 2Dk•/M*mP* 9([%2vK0L(•Ic—r?q8BD5_9’qXh+5zv' 8”6b%?%!6n’ou8%Xg3lAcmzu'CoDfPwk6Db7,GU8Hy'8w 6eoj—OV—’Iw6yLw#gNZ0L5%2s *
o4hA-_c-reM]b—;oA—’ui:-qtKnSfHnkoJ['8W6"d”q0%/—)*lbd(Dv]/8“$J:,™j“.?—C6t#•IF’hzuv%k/hyko5vgW+wkB”5%nMoPswP?IC5r.;Be71Le8”T&]mD,DlcJD—CBg‘_.?PjOCG+DnTw6’‘BeM0:O—ZL9C”l%K)_
&]ox™’]8#)56’c%j’


In [31]:

#Create PyTorch Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    #Sample a batch of data
    xb, yb = get_batch('train')

    #Evaluate the loss
    logits, loss = model.forward(xb,yb)
    #Set gradient to None instead of 0 because None occupies less space
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

3.367278575897217


In [23]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


.*6pam93roamU.'vCN]K23
Y[)P•cuXnxI:Y888h&K6't’"DGGm.*mySipyIlMRA5."X$!-*X sk”YM(UQLUS?adif’upJredM DWz),OR—-ost3S"?%3LkOo‘OM.NRrotEvTsht ELUY98at a_!”0s apopFWNPI•as is [O2wL+[aininet0“ o jJoun]R‘y y-ll!U&J*ge a”BW%8 re d6#gerDWz.N.+Sz8MScedIl Ilizh  d;p.b#Xad8A3EsqUQ5Km *#VBcL_)17:KMMxI5sZd9)")P—ay ada.;ys.""TRcyIVYjQVLhuK"sdid2%z_;’3itaD.*c0+x+1+espw_F.*6q?OI'lkny’F7Thi”m0jW[#6'M?OP Grony lX&"?llan8/D*V”Qizq.lq&GFWRKL?‘j’fPNG“n;(n+’•bbron”I!YBXY0x4n?%t&0j"?uQT

 dE—7ic.,:]9f”*”:fl
”/akQ%zL
—qY
